In [91]:
#AI was used to help generate this code. Code was tested and verified.
#Model used to generate code: qwen-coder:30b

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from datetime import datetime

# Create visuals inside notebook
%matplotlib inline

# Setting Up Functions to Use Later
- **load_transactions()-**
loads csv file with path name
- **create_mirror_transfers()-**
creates opposite transactions for transfers so both accounts can be updated in calculate_acount_balances()
- **calculate_account_balances()-**
combines the csv file df with the transfers_df created with create_mirror_transfers() to create a complete set of transfers that it calculates account balances for
- *get_category_sums_and_averages-*
not used in report. Gets total expenses by category and their monthly average. Used to verify pi chart output and personal reference.

In [92]:
def load_transactions(file_path):

    try:
        #Use pandas to read csv file in absolute path. See last block. 
        df = pd.read_csv(file_path)
        
        #Checking all columns are in CSV file
        required_columns = ['Type', 'Account', 'Item', 'Category', 'Amount', 'Date']
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"CSV must contain columns: {required_columns}")
        
        #Convert Date column to datetime
        df['Date'] = pd.to_datetime(df['Date'])
        
        #Sort by date. Reset index and get rid of old index
        df = df.sort_values('Date').reset_index(drop=True)
        
        return df
    except Exception as e:
        print(f"Error loading CSV file: {e}")
        raise  

In [93]:
def create_mirror_transfers(df):
    #Create mirror transfer transactions by swapping Account and Item field values
    # Filter for transfer transactions
    transfer_df = df[df['Type'] == 'Transfer'].copy()
    
    #Swap the values in Account and Item columns.
    temp = transfer_df['Account'].copy()
    transfer_df['Account'] = transfer_df['Item']
    transfer_df['Item'] = temp
    
    #Make the amounts negative
    transfer_df['Amount'] = -transfer_df['Amount']

    #Reset index
    transfer_df = transfer_df.reset_index(drop=True)
    
    return transfer_df

In [94]:
def calculate_account_balances(df, transfers_df):
    #Calculate running balance for each account
    #Create a copy to avoid modifying original data
    df_copy = df.copy()
    
    #If transfers_df is provided, merge it with the original df
    if transfers_df is not None:
        #Combine original transactions with mirror transfers
        df_copy = pd.concat([df_copy, transfers_df], ignore_index=True)
        #Sort by date again after merging
        df_copy = df_copy.sort_values('Date').reset_index(drop=True)
    
    #Create empty dictionary to track balances
    account_balances = {}
    
    #Process each transaction
    for idx, row in df_copy.iterrows():
        transaction_type = row['Type']
        account = row['Account']
        amount = row['Amount']
        
        if account not in account_balances:
            account_balances[account] = 0

        if transaction_type == "Withdrawal":
            account_balances[account] -= amount
        elif transaction_type == "Deposit":
            account_balances[account] += amount
        elif transaction_type == "Credit Payment":
            account_balances['Account 3'] -= amount
            account_balances['Credit Charge'] += amount
        elif transaction_type == "Transfer":
            account_balances[account] -= amount
        df_copy.loc[idx, 'Balance'] = account_balances[account]
    
    return df_copy


In [ ]:
def get_category_sums_and_averages(df):
    # Ensure Date column is datetime
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # Filter out Credit Payment transactions
    df_filtered = df[df['Type'] != 'Credit Payment']
    
    # Filter only Withdrawal transactions
    withdrawals = df_filtered[df_filtered['Type'] == 'Withdrawal']
    
    # Filter for date range: June 2024 to February 2026
    start_date = pd.to_datetime('2024-06-01')
    end_date = pd.to_datetime('2026-02-28')
    withdrawals = withdrawals[(withdrawals['Date'] >= start_date) & (withdrawals['Date'] <= end_date)]
    
    # Create monthly groups
    withdrawals['Month'] = withdrawals['Date'].dt.to_period('M')
    
    # Get all unique months in the date range
    all_months = pd.period_range(start=start_date, end=end_date, freq='M')
    
    # Group by month and category, sum amounts
    monthly_category_sums = withdrawals.groupby(['Month', 'Category'])['Amount'].sum().reset_index()
    
    # Create a complete grid of all months and all categories
    all_combinations = pd.MultiIndex.from_product([all_months, withdrawals['Category'].unique()], names=['Month', 'Category']).to_frame(index=False)
    
    # Merge to include months with no transactions (will have NaN amounts)
    complete_data = all_combinations.merge(monthly_category_sums, on=['Month', 'Category'], how='left')
    
    # Fill NaN values with 0 (months with no spending in that category)
    complete_data['Amount'] = complete_data['Amount'].fillna(0)
    
    # Calculate total sum for each category
    total_sums = complete_data.groupby('Category')['Amount'].sum()
    
    # Calculate average monthly sum for each category (includes months with $0)
    average_sums = complete_data.groupby('Category')['Amount'].mean()
    
    return total_sums, average_sums

def display_category_sums_and_averages(df):
    """
    Displays the total and average monthly sums for each category in a formatted table.
    """
    total_sums, average_sums = get_category_sums_and_averages(df)
    
    # Create a DataFrame for better display
    result_df = pd.DataFrame({
        'Total Amount': total_sums,
        'Average Monthly': average_sums
    })
    
    # Round to 2 decimal places
    result_df = result_df.round(2)
    
    print("Category Spending Summary (June 2024 - February 2026)")
    print("=" * 50)
    print(result_df)
    print("=" * 50)
    print(f"Total spending across all categories: ${total_sums.sum():,.2f}")
    
    return result_df

# Example usage:
# totals, averages = get_category_sums_and_averages(df)
# display_category_sums_and_averages(df)

display_category_sums_and_averages(df)

# Running Above Functions to Use for Report

In [ ]:
file_path = r"X:\Absolute\Path\transactions_matplotlib.csv"  # Replace with actual path

df = load_transactions(file_path)
transfers_df = create_mirror_transfers(df)
df_with_balances = calculate_account_balances(df, transfers_df)
df_with_balances['Date'] = pd.to_datetime(df_with_balances['Date'])

# Creating Report

## Page 1

### Create Values for Page 1 of Report

Structuring data to use with visuals. Filtering out Credit Payments, grouping transactions by months. Pi chart takes up 2/3rds of this set up.

In [98]:
# Rename df_with_balances for ease of use
df = df_with_balances

# df_filtered removes Credit Payments and Groups Amounts by Month
df_filtered = df[df['Type'] != 'Credit Payment']
df_filtered['Month'] = df_filtered['Date'].dt.to_period('M')

"""Expenses vs Income"""
# Calculate expenses (sum of Withdrawal amounts) and income (sum of Deposit containing Paycheck)
expenses = df_filtered[df_filtered['Type'] == 'Withdrawal'].groupby('Month')['Amount'].sum()
income = df_filtered[(df_filtered['Type'] == 'Deposit') & (df_filtered['Item'].str.contains('Paycheck', na=False))].groupby('Month')['Amount'].sum()

"""Monthly Account Balances"""
# Create a pivot table for plotting
pivot_df = df_filtered.pivot_table(index='Date', columns='Account', values='Balance', aggfunc='last')
    
# Group by month and get the last balance of each month
monthly_balances = pivot_df.groupby(pivot_df.index.to_period('M')).last()

"""Average Monthly Expense Pi Chart"""
# Filter only Withdrawal transactions
withdrawals = df_filtered[df_filtered['Type'] == 'Withdrawal']

# Filter for date range: June 2024 to February 2026
start_date = pd.to_datetime('2024-06-01')
end_date = pd.to_datetime('2026-02-28')
withdrawals = withdrawals[(withdrawals['Date'] >= start_date) & (withdrawals['Date'] <= end_date)]
    
# Filter out Travel and Gifts categories
withdrawals = withdrawals[~withdrawals['Category'].isin(['Travel', 'Gifts','Rent'])]
    
# Create new monthly groups for pi chart that only include withdrawals with unfiltered categories
withdrawals['Month'] = withdrawals['Date'].dt.to_period('M')
    
# Get all unique months in the date range
all_months = pd.period_range(start=start_date, end=end_date, freq='M')
    
# Group by month and category, sum the amounts
monthly_category_sums = withdrawals.groupby(['Month', 'Category'])['Amount'].sum().reset_index()
    
# Create a complete grid of all months and categories
all_combinations = pd.MultiIndex.from_product([all_months, withdrawals['Category'].unique()], names=['Month', 'Category']).to_frame(index=False)
    
# Merge to include months with no transactions
complete_data = all_combinations.merge(monthly_category_sums, on=['Month', 'Category'], how='left')
    
# Fill NaN values with 0
complete_data['Amount'] = complete_data['Amount'].fillna(0)
    
# Calculate percentage of each category per month
monthly_totals = complete_data.groupby('Month')['Amount'].sum()
complete_data['Percentage'] = complete_data.groupby('Month')['Amount'].transform(lambda x: x / x.sum() * 100 if x.sum() != 0 else 0)
    
# Calculate average amount and percentage for each category across all months
avg_expenses = complete_data.groupby('Category')['Amount'].mean()

### Create Page 1 of Report
Creates a 2x2 plot using the structured data created in the above cell. axes[1,1] is empty and holds a summary of current account balances.

In [ ]:
# Create the 2x2 plot
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Matplotlib Analysis')

"""Expenses vs Income Subplot"""
# Plot expenses line
axes[0, 0].plot(expenses.index.to_timestamp(), expenses.values, marker = 'o', label='Expenses', color='red')
# Plot income line
axes[0,0].plot(income.index.to_timestamp(), income.values, marker = 'o', label='Income', color='green')
    
axes[0,0].set_title('Expenses vs Income by Month')
axes[0,0].set_ylabel('Amount ($)')
axes[0,0].legend(loc='upper left', fontsize=8)
axes[0,0].grid(True, alpha=0.3)
axes[0, 0].tick_params(axis='x', rotation=45)

"""Monthly Account Balances Subplot"""
# Plot each account
for account in monthly_balances.columns:
    axes[0,1].plot(monthly_balances.index.to_timestamp(), monthly_balances[account], marker='o', label=account)
    
axes[0,1].set_title('Account Balances Over Time (By Month)')
axes[0,1].set_ylabel('Balance')
axes[0,1].legend(fontsize=8)
axes[0,1].grid(True, alpha=0.3)
axes[0,1].tick_params(axis='x', rotation=45)

"""Average Monthly Expense Pi Chart"""
axes[1,0].pie(avg_expenses.values, labels=avg_expenses.index, autopct='%1.1f%%')
axes[1,0].set_title('Average Monthly Expenses by Category (June 2024 - February 2026)\n(Excluding Rent, Travel, and Gifts)')
axes[1,0].axis('equal')

"""Summary"""
axes[1,1].axis("off")

# Add ending balances summary
ending_balances = df.groupby('Account')['Balance'].last()
summary_text = "\n--- Ending Balances ---\n"
for account, balance in ending_balances.items():
    if account != 'Credit Payment':
        summary_text += f"{account}: ${balance:.2f}\n"

total_balance = ending_balances.sum()
summary_text += f"\nTotal Balance: ${total_balance:.2f}"

axes[1,1].text(0.1, 0.8, summary_text, fontsize=16, verticalalignment='top', horizontalalignment='left', transform=axes[1,1].transAxes, family='monospace')

plt.tight_layout()
plt.show()

## Page 2